In [1]:
!wget "https://storage.googleapis.com/kaggle-competitions-data/kaggle-v2/21154/1243559/compressed/tfrecords-jpeg-192x192.zip?GoogleAccessId=web-data@kaggle-161607.iam.gserviceaccount.com&Expires=1784469965&Signature=FzcStMZORiwZIdObb7%2FH3xXcxw8DMEZ4Yb%2FXwdbg8fq%2Fq0rYUVww4PGkLGrp3xLCexe8qkFa4V3itCU3M58oyjkNLlgtmE1kP0V%2BBf2iA71GwIiH%2BpPaPAW%2B99xYS598z32cvWP2YjtDDBCMH%2Ff8bQccTUvE2441tzlloNGI9TzPwEPt3ZJlTRH82l%2B6iWr26cc2Tc0h5XtnmPmZ20rIEK5WDYdUQffzKWhqdLpsrXwKwTiNQVGKgCVdmJkk0j14AKo1224Pq7iS8N4F%2BjKrCeQ6SADVxKZFoIjJxfTqrRnFlP7enwxvrS40jdr3re9BFRWAnMEgdds7dnMByctb%2Bw%3D%3D&response-content-disposition=attachment%3B+filename%3Dtfrecords-jpeg-192x192.zip" -o "tfrecords-jpeg-192x192.zip"

The destination name is too long (547), reducing to 236
--2026-07-16 15:01:59--  https://storage.googleapis.com/kaggle-competitions-data/kaggle-v2/21154/1243559/compressed/tfrecords-jpeg-192x192.zip?GoogleAccessId=web-data@kaggle-161607.iam.gserviceaccount.com&Expires=1784469965&Signature=FzcStMZORiwZIdObb7%2FH3xXcxw8DMEZ4Yb%2FXwdbg8fq%2Fq0rYUVww4PGkLGrp3xLCexe8qkFa4V3itCU3M58oyjkNLlgtmE1kP0V%2BBf2iA71GwIiH%2BpPaPAW%2B99xYS598z32cvWP2YjtDDBCMH%2Ff8bQccTUvE2441tzlloNGI9TzPwEPt3ZJlTRH82l%2B6iWr26cc2Tc0h5XtnmPmZ20rIEK5WDYdUQffzKWhqdLpsrXwKwTiNQVGKgCVdmJkk0j14AKo1224Pq7iS8N4F%2BjKrCeQ6SADVxKZFoIjJxfTqrRnFlP7enwxvrS40jdr3re9BFRWAnMEgdds7dnMByctb%2Bw%3D%3D&response-content-disposition=attachment%3B+filename%3Dtfrecords-jpeg-192x192.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 74.125.137.207, 142.250.101.207, 142.250.141.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|74.125.137.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Le

In [3]:
import shutil

# Extract the entire ZIP file into a specific directory
shutil.unpack_archive("tfrecords-jpeg-192x192.zip", "tfrecords-jpeg-192x192")

In [14]:
import tensorflow as tf

In [5]:
print("TF version:", tf.__version__)
strategy = tf.distribute.get_strategy()
print("GPU:", tf.config.list_physical_devices('GPU'))
AUTOTUNE = tf.data.AUTOTUNE

TF version: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [24]:
IMAGE_SIZE = [192,192]
NUM_CLASSES = 104
BATCH_SIZE = 16

folder_path = '/content/tfrecords-jpeg-192x192'

TRAIN_FILES = [
    os.path.join(folder_path, "train", f)
    for f in os.listdir(os.path.join(folder_path, "train"))
    if f.endswith(".tfrec")
]

VAL_FILES = [
    os.path.join(folder_path, "val", f)
    for f in os.listdir(os.path.join(folder_path, "val"))
    if f.endswith(".tfrec")
]

print("files:", len(TRAIN_FILES), len(VAL_FILES))

files: 16 16


In [10]:
import re
import numpy as np

# Decode a JPEG string into a normalized float image
def decode_image(image_data):
    image = tf.image.decode_jpeg(image_data, channels=3)
    image = tf.cast(image, tf.float32)
    image = tf.reshape(image, [*IMAGE_SIZE, 3])
    return image

# Parse a labeled example (image + class)
def read_labeled(example):
    fmt = {"image": tf.io.FixedLenFeature([], tf.string),
           "class": tf.io.FixedLenFeature([], tf.int64)}
    ex = tf.io.parse_single_example(example, fmt)
    return decode_image(ex["image"]), tf.cast(ex["class"], tf.int32)

# Parse an unlabeled example (image + id)
def read_unlabeled(example):
    fmt = {"image": tf.io.FixedLenFeature([], tf.string),
           "id": tf.io.FixedLenFeature([], tf.string)}
    ex = tf.io.parse_single_example(example, fmt)
    return decode_image(ex["image"]), ex["id"]

# Count images from the number encoded in each filename
def count_items(filenames):
    n = [int(re.compile(r"-([0-9]*)\.").search(f).group(1)) for f in filenames]
    return int(np.sum(n))

In [11]:
# Build a TFRecord dataset, optionally deterministic
def load_dataset(filenames, labeled=True, ordered=False):
    ds = tf.data.TFRecordDataset(filenames, num_parallel_reads=AUTOTUNE)
    if ordered:
        opt = tf.data.Options(); opt.deterministic = True
        ds = ds.with_options(opt)
    ds = ds.map(read_labeled if labeled else read_unlabeled, num_parallel_calls=AUTOTUNE)
    return ds

# Training set: shuffled, repeated, batched
def get_training_dataset():
    ds = load_dataset(TRAIN_FILES, labeled=True)
    return ds.repeat().shuffle(2048).batch(BATCH_SIZE).prefetch(AUTOTUNE)

# Validation set: ordered, batched
def get_validation_dataset():
    return load_dataset(VAL_FILES, labeled=True, ordered=True).batch(BATCH_SIZE).prefetch(AUTOTUNE)

NUM_TRAIN, NUM_VAL= count_items(TRAIN_FILES), count_items(VAL_FILES)
STEPS_PER_EPOCH = NUM_TRAIN // BATCH_SIZE
print("train/val:", NUM_TRAIN, NUM_VAL, "| steps/epoch:", STEPS_PER_EPOCH)

train/val: 12753 3712 | steps/epoch: 797


In [18]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.20),
    tf.keras.layers.RandomContrast(0.20),
    tf.keras.layers.RandomBrightness(0.20),
], name="augmentation")

In [66]:
from tensorflow.keras import layers, Model
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input

base_model = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(192,192,3)
)

base_model.trainable = False

inputs = tf.keras.Input(shape=(192,192,3))

x = data_augmentation(inputs)
x = tf.keras.applications.efficientnet.preprocess_input(x)

x = base_model(x, training=False)

x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.4)(x)

x = Dense(256, activation='swish')(x)
x = Dropout(0.4)(x)

outputs = tf.keras.layers.Dense(
    NUM_CLASSES,
    activation="softmax"
)(x)

model = tf.keras.Model(inputs, outputs)

In [67]:
model.summary()

Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_15 (InputLayer)     │ (None, 192, 192, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ augmentation (Sequential)       │ (None, 192, 192, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 6, 6, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_7      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 104)            │        26,728 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,404,235 (16.80 MB)

 Trainable params: 354,664 (1.35 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [68]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        "accuracy",
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=5,
            name="top5_acc"
        )
    ]
)

In [69]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "best_efficientnet.keras",
        monitor="val_accuracy",
        save_best_only=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )
]

In [70]:
history = model.fit(
    get_training_dataset(),
    validation_data=get_validation_dataset(),
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_steps=NUM_VAL // BATCH_SIZE,
    epochs=15,
    callbacks=callbacks
)

Epoch 1/15
797/797 ━━━━━━━━━━━━━━━━━━━━ 59s 59ms/step - accuracy: 0.4617 - loss: 2.2406 - top5_acc: 0.7230 - val_accuracy: 0.7136 - val_loss: 1.0852 - val_top5_acc: 0.9235 - learning_rate: 0.0010
Epoch 2/15
797/797 ━━━━━━━━━━━━━━━━━━━━ 44s 56ms/step - accuracy: 0.6249 - loss: 1.4172 - top5_acc: 0.8725 - val_accuracy: 0.7643 - val_loss: 0.8654 - val_top5_acc: 0.9464 - learning_rate: 0.0010
Epoch 3/15
797/797 ━━━━━━━━━━━━━━━━━━━━ 44s 55ms/step - accuracy: 0.6664 - loss: 1.2283 - top5_acc: 0.8988 - val_accuracy: 0.7931 - val_loss: 0.7866 - val_top5_acc: 0.9472 - learning_rate: 0.0010
Epoch 4/15
797/797 ━━━━━━━━━━━━━━━━━━━━ 44s 56ms/step - accuracy: 0.6827 - loss: 1.1411 - top5_acc: 0.9115 - val_accuracy: 0.7934 - val_loss: 0.7478 - val_top5_acc: 0.9502 - learning_rate: 0.0010
Epoch 5/15
797/797 ━━━━━━━━━━━━━━━━━━━━ 43s 54ms/step - accuracy: 0.6956 - loss: 1.0859 - top5_acc: 0.9195 - val_accuracy: 0.7980 - val_loss: 0.7355 - val_top5_acc: 0.9534 - learning_rate: 0.0010
Epoch 6/15
797/797 ━

In [77]:
model.get_layer('dense_7').trainable = False

In [78]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=5,
            name="top5_acc"
        )
    ]
)

In [79]:
model.summary()

Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_15 (InputLayer)     │ (None, 192, 192, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ augmentation (Sequential)       │ (None, 192, 192, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 6, 6, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_7      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 104)            │        26,728 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,404,235 (16.80 MB)

 Trainable params: 26,728 (104.41 KB)

 Non-trainable params: 4,377,507 (16.70 MB)

In [80]:
history_ft = model.fit(
    get_training_dataset(),
    validation_data=get_validation_dataset(),
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_steps=NUM_VAL // BATCH_SIZE,
    epochs=10,
    callbacks=callbacks
)

Epoch 1/10
797/797 ━━━━━━━━━━━━━━━━━━━━ 56s 55ms/step - accuracy: 0.7692 - loss: 0.7939 - top5_acc: 0.9520 - val_accuracy: 0.8386 - val_loss: 0.6167 - val_top5_acc: 0.9588 - learning_rate: 1.0000e-05
Epoch 2/10
796/797 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.7651 - loss: 0.8006 - top5_acc: 0.9524
Epoch 2: ReduceLROnPlateau reducing learning rate to 1.9999999494757505e-06.
797/797 ━━━━━━━━━━━━━━━━━━━━ 41s 51ms/step - accuracy: 0.7712 - loss: 0.7777 - top5_acc: 0.9555 - val_accuracy: 0.8386 - val_loss: 0.6167 - val_top5_acc: 0.9588 - learning_rate: 1.0000e-05
Epoch 3/10
797/797 ━━━━━━━━━━━━━━━━━━━━ 39s 50ms/step - accuracy: 0.7698 - loss: 0.7984 - top5_acc: 0.9526 - val_accuracy: 0.8378 - val_loss: 0.6167 - val_top5_acc: 0.9588 - learning_rate: 2.0000e-06
Epoch 4/10
797/797 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.7693 - loss: 0.7653 - top5_acc: 0.9548
Epoch 4: ReduceLROnPlateau reducing learning rate to 3.999999989900971e-07.
797/797 ━━━━━━━━━━━━━━━━━━━━ 42s 52ms/step - a